# STAT 5243 Project 1: WallStreetBets Data Pipeline and Exploratory Analytics

**Team:** Zeming Liang, Zuer Weng, Duoli Chen, Isaac Beers

**Reviewer Contract:** This notebook is written so a professor can grade from **one PDF and one code file**.
- Single report PDF target: `Project Deliverables/Report/STAT5243_Project1_Team2_Final.pdf`
- Single canonical workflow code file: `Project Deliverables/Code Files/Project1_Full_Workflow_Code.ipynb`
- Backup automation script: `Project Deliverables/Code Files/full_workflow.py`


## 1. Introduction and Dataset Description

This project analyzes **53,187 posts** from `r/wallstreetbets` during the 2020-2021 meme-stock period. The dataset is complex because it combines:
- structured engagement variables (`score`, `comms_num`, timestamps, URLs),
- unstructured language data (`title`, `body`) with markdown/slang/noise,
- temporal behavior with regime-shift dynamics.

Quantitative complexity indicators include:
- `missing_body_pct = 53.49%` (structural missingness),
- heavy-tailed engagement distributions with viral outliers,
- mixed raw/derived feature requirements for modeling readiness.

These characteristics justify advanced-level cleaning, EDA, and feature-engineering work.


In [ ]:
import pandas as pd

raw = pd.read_csv('../Datasets/reddit_wsb.csv')
clean = pd.read_csv('../Datasets/reddit_wsb_cleaned.csv')

print('raw_shape:', raw.shape)
print('clean_shape:', clean.shape)
print('raw_columns:', list(raw.columns)[:8], '...')


## 2. Data Acquisition Methodology

The dataset was acquired from a **public repository (Kaggle)** and originally collected from Reddit via **PRAW**. We preserve the raw file unchanged and perform deterministic transformation into the cleaned analytical dataset.

### Acquisition workflow
1. Download public dataset and preserve immutable raw input.
2. Validate schema and coverage before transformation.
3. Apply deterministic cleaning/preprocessing workflows.
4. Export cleaned dataset and evidence artifacts (figures + JSON) for full auditability.

### Why one-source can still be advanced
Although this project uses a single primary source, the source itself has significant real-world quality challenges (missing text, skewed targets, mixed structures), which creates advanced-level data engineering and interpretation work.

Evidence sources:
- `Project Workspace/Supporting Materials/Report Sources/artifacts/json/02_cleaning_policy_metrics.json`
- `Project Workspace/Supporting Materials/Report Sources/artifacts/json/02_cleaning_fig_01_missingness_diagnostics.json`


In [ ]:
# Integrity checks for acquisition -> processing continuity
print('row_count_raw:', len(raw))
print('row_count_clean:', len(clean))
print('same_row_count:', len(raw) == len(clean))


## 3. Cleaning and Preprocessing Steps

Cleaning is implemented as policy-driven transformation with explicit handling of each required inconsistency class.

### 3.1 Type consistency and formatting normalization
- Datetime coercion and temporal normalization.
- Text normalization for markdown/URL cleanup.
- Uniform transformed fields (`score_log`, `comms_num_log`, normalized variants).

### 3.2 Duplicate handling
- Duplicate checks on IDs and full rows.
- Result: no duplicate IDs and no duplicate full rows.

### 3.3 Missing data handling
- Structural missingness retained with explicit indicator variables instead of naive row deletion.
- Body missingness quantified and documented.

### 3.4 Outlier strategy
- Heavy-tailed variables are stabilized using `log1p` transforms.
- Diagnostics evaluate post-transform behavior and leverage influence.

### 3.5 Scaling and encoding
- Z-score and min-max scaling for numerical comparability.
- Categorical encoding and grouped post-type features for modeling inputs.

### Preprocessing decision table (issue -> method -> justification)
| Issue | Method | Justification | Evidence |
|---|---|---|---|
| Incorrect/variable types | Strict coercion + normalization | Prevent invalid downstream feature extraction | `02_cleaning_policy_metrics.json` |
| Duplicates | ID + full-row audit | Avoid silent inflation bias | `02_cleaning_policy_metrics.json` |
| Missing text | Structural treatment + indicators | Preserve representativeness while modeling missingness explicitly | `02_cleaning_fig_01_missingness_diagnostics.json` |
| Outliers | `log1p`, robust diagnostics | Reduce extreme leverage from viral tails | `02_cleaning_fig_03_outlier_visualization.json` |
| Scale mismatch | z-score + min-max features | Support comparable model inputs | `02_cleaning_fig_07_scaling_comparison.json` |


In [ ]:
# Required cleaning metrics (quick reproducibility excerpt)
missing_body_pct = raw['body'].isna().mean() * 100
dup_id = raw['id'].duplicated().sum()
dup_row = raw.duplicated().sum()

print('missing_body_pct:', round(missing_body_pct, 4))
print('duplicate_id_count:', int(dup_id))
print('duplicate_full_row_count:', int(dup_row))


### Core Cleaning Figures


![02_cleaning_fig_01_missingness_diagnostics.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_01_missingness_diagnostics.png)


![02_cleaning_fig_03_outlier_visualization.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_03_outlier_visualization.png)


![02_cleaning_fig_07_scaling_comparison.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_07_scaling_comparison.png)


## 4. Exploratory Data Analysis (EDA)

The EDA stage combines descriptive views and inferential statistics to uncover nuanced behavior.

### 4.1 Descriptive + relational patterns
- Distributional skew and transformed distribution stabilization.
- Engagement coupling between score and comments.
- Temporal and post-type variation patterns.

### 4.2 Advanced statistical diagnostics
- Spearman correlation (`rho = 0.7852`, `p << 0.001`) for monotonic engagement dependence.
- Kruskal-Wallis post-type test (`H = 5855.658`, `p << 0.001`) for group differences.
- Outlier-rate diagnostics (`|z| >= 3` rate ~ `0.173%`).

### 4.3 Interpretation
The data shows event-driven attention clustering: most posts are low-engagement, while rare viral spikes dominate influence, requiring robust transforms and caution in practical significance interpretation.


In [ ]:
import json
from pathlib import Path

stats_path = Path('../../Project Workspace/Supporting Materials/Report Sources/artifacts/json/03_eda_advanced_stats.json')
eda_stats = json.loads(stats_path.read_text())
print(eda_stats)


### Core EDA Figures


![03_eda_fig_01_score_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_01_score_distribution.png)


![03_eda_fig_03_score_log_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_03_score_log_distribution.png)


![03_eda_fig_10_score_comments_scatter.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_10_score_comments_scatter.png)


![03_eda_fig_19_post_type_median_iqr.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_19_post_type_median_iqr.png)


![03_eda_fig_17_daily_outlier_count.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_17_daily_outlier_count.png)


## 5. Feature Engineering Process and Justification

Feature engineering extends baseline engagement predictors with text-derived signals to evaluate incremental predictive utility.

### 5.1 Engineered features
- Sentiment score from title lexicon rules.
- Topic-probability features via LDA topic modeling.
- Encoded/normalized predictors from preprocessing stage.

### 5.2 Ablation (before/after utility)
| Model | Feature Count | ROC-AUC | Avg Precision | F1@0.5 |
|---|---:|---:|---:|---:|
| Baseline | 6 | 0.999984 | 0.999710 | 0.876950 |
| Baseline + Sentiment | 7 | 0.999984 | 0.999710 | 0.876950 |
| Baseline + Sentiment + Topics | 12 | 0.999975 | 0.999546 | 0.884615 |

Interpretation:
- Sentiment alone produced negligible lift under this target definition.
- Topic features improved threshold performance (`+0.007666` absolute F1 lift), showing context adds practical decision utility.

### 5.3 Limitations
- Virality label is score-derived and may induce leakage-like ease.
- Lexicon sentiment has sarcasm/context limitations.
- Topic features improve performance but reduce direct single-post interpretability.

Source JSON: `Project Workspace/Supporting Materials/Report Sources/artifacts/json/05_feature_ablation_table.json`


In [ ]:
ablation_path = Path('../../Project Workspace/Supporting Materials/Report Sources/artifacts/json/05_feature_ablation_table.json')
ablation = json.loads(ablation_path.read_text())
for row in ablation['models']:
    print(row)


### Workflow Architecture (One Code File Visibility)
The full workflow is implemented in one canonical script:
- `../Code Files/Project1_Full_Workflow_Code.ipynb` (canonical review code file)
- `../Code Files/full_workflow.py` (backup script)

Pipeline stages:
1. Load and validate raw schema.
2. Cleaning/preprocessing transforms and feature preparation.
3. EDA figure/stat generation.
4. Feature diagnostics and ablation evaluation.
5. Export artifacts and summary JSON.

CLI example:
```bash
jupyter nbconvert --to notebook --execute "Project Deliverables/Code Files/Project1_Full_Workflow_Code.ipynb" --output /tmp/Project1_Full_Workflow_Code.executed.ipynb
python3 "Project Deliverables/Code Files/full_workflow.py" --raw "Project Deliverables/Datasets/reddit_wsb.csv" --out-dir "Project Workspace/Supporting Materials/Generated Outputs" --sample-size 20000
```


### Core Feature Figures


![04_feature_fig_01_sentiment_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_01_sentiment_distribution.png)


![04_feature_fig_04_roc_curve.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_04_roc_curve.png)


![04_feature_fig_05_pr_curve.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_05_pr_curve.png)


![04_feature_fig_06_confusion_matrix.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_06_confusion_matrix.png)


![05_feature_ablation_comparison.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/05_feature_ablation_comparison.png)


## 6. Summary of Key Findings

- Data complexity is high due to mixed structures, structural missingness, and heavy-tailed engagement outcomes.
- Cleaning addressed all major inconsistency types with explicit, auditable policy choices.
- EDA confirmed strong engagement dependence and statistically significant post-type effects.
- Feature engineering showed measurable threshold-lift from topic features.
- The one-file workflow is reproducible and supported by figure/JSON evidence artifacts.

Conclusion: the project satisfies full pipeline expectations with advanced-level evidence depth and transparent reproducibility.


### Figure-to-Finding Traceability (Core Claims)

| Claim ID | Core finding | Figure evidence | JSON evidence |
|---|---|---|---|
| C1 | Structural missingness is high (~53.49% body missing) and must be handled explicitly. | `02_cleaning_fig_01_missingness_diagnostics.png` | `Project Workspace/Supporting Materials/Report Sources/artifacts/json/02_cleaning_fig_01_missingness_diagnostics.json` |
| C2 | Engagement metrics are monotonic with strong dependence between score and discussion volume. | `03_eda_fig_10_score_comments_scatter.png` | `Project Workspace/Supporting Materials/Report Sources/artifacts/json/03_eda_advanced_stats.json` |
| C3 | Score distribution has heavy tails with a small extreme outlier fraction. | `03_eda_fig_01_score_distribution.png`, `03_eda_fig_17_daily_outlier_count.png` | `Project Workspace/Supporting Materials/Report Sources/artifacts/json/03_eda_advanced_stats.json` |
| C4 | Feature enrichment with topics improves threshold utility beyond baseline/sentiment-only. | `05_feature_ablation_comparison.png` | `Project Workspace/Supporting Materials/Report Sources/artifacts/json/05_feature_ablation_table.json` |
| C5 | Classifier discrimination is high and supported by ROC/PR diagnostics. | `04_feature_fig_04_roc_curve.png`, `04_feature_fig_05_pr_curve.png` | `Project Workspace/Supporting Materials/Report Sources/artifacts/json/04_feature_fig_04_roc_curve.json` |


## 7. Challenges Faced and Future Recommendations

### Challenges
- Structural missingness in body text and noisy social-media language.
- Heavy tails and rare viral events complicating stable modeling.
- Label-definition caveat due to score-derived virality target.

### Recommendations
- Move toward forward-looking targets and temporal validation splits.
- Add richer contextual NLP and sarcasm-aware sentiment approaches.
- Integrate external market variables for stronger cross-domain interpretation.


## 8. GitHub Repository Link

- https://github.com/ZemingLiang/STAT-5243-Project-1-Team-2


## 9. Each Member's Contribution

- **Zeming Liang**: Data cleaning and preprocessing; creation/normalization of `reddit_wsb.csv`, `reddit_wsb_cleaned.csv`, and `Cleaning-and-Preprocessing.ipynb`; final repo organization and PNG/JSON/output integration.
- **Zuer Weng**: EDA analysis, advanced statistical diagnostics, and EDA branch artifacts.
- **Duoli Chen**: Report-writing narrative sections (introduction, acquisition, summary, challenges) and narrative refinement.
- **Isaac Beers**: Feature engineering workflow, viral-classification diagnostics, and model interpretation components.


## Appendix A. Full Evidence Figure Index (Inside Same PDF)

This appendix contains additional supporting visuals so the report remains one self-contained grading artifact.


| Figure ID | Section | Purpose |
|---|---|---|
| `02_cleaning_fig_02_score_by_body_presence.png` | Cleaning | Supporting diagnostic evidence |
| `02_cleaning_fig_04_influence_plot.png` | Cleaning | Supporting diagnostic evidence |
| `02_cleaning_fig_05_qq_normality.png` | Cleaning | Supporting diagnostic evidence |
| `02_cleaning_fig_06_spearman_heatmap.png` | Cleaning | Supporting diagnostic evidence |
| `03_eda_fig_02_comms_distribution.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_04_comms_log_distribution.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_05_title_length_distribution.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_06_hour_distribution.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_07_post_type_lumped_bar.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_08_post_type_bar.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_09_day_of_week_bar.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_11_title_score_scatter.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_12_score_hour_trend.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_13_comments_hour_trend.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_14_score_dow_trend.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_15_daily_score_trend.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_16_corr_heatmap.png` | EDA | Supporting diagnostic evidence |
| `03_eda_fig_18_hour_day_heatmap.png` | EDA | Supporting diagnostic evidence |
| `04_feature_fig_02_topic_entropy_distribution.png` | Feature Engineering | Supporting diagnostic evidence |
| `04_feature_fig_03_dominant_topic_distribution.png` | Feature Engineering | Supporting diagnostic evidence |
| `04_feature_fig_07_pred_prob_distribution.png` | Feature Engineering | Supporting diagnostic evidence |
| `04_feature_fig_08_top_coefficients.png` | Feature Engineering | Supporting diagnostic evidence |


![02_cleaning_fig_02_score_by_body_presence.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_02_score_by_body_presence.png)


![02_cleaning_fig_04_influence_plot.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_04_influence_plot.png)


![02_cleaning_fig_05_qq_normality.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_05_qq_normality.png)


![02_cleaning_fig_06_spearman_heatmap.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/02_cleaning_fig_06_spearman_heatmap.png)


![03_eda_fig_02_comms_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_02_comms_distribution.png)


![03_eda_fig_04_comms_log_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_04_comms_log_distribution.png)


![03_eda_fig_05_title_length_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_05_title_length_distribution.png)


![03_eda_fig_06_hour_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_06_hour_distribution.png)


![03_eda_fig_07_post_type_lumped_bar.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_07_post_type_lumped_bar.png)


![03_eda_fig_08_post_type_bar.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_08_post_type_bar.png)


![03_eda_fig_09_day_of_week_bar.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_09_day_of_week_bar.png)


![03_eda_fig_11_title_score_scatter.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_11_title_score_scatter.png)


![03_eda_fig_12_score_hour_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_12_score_hour_trend.png)


![03_eda_fig_13_comments_hour_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_13_comments_hour_trend.png)


![03_eda_fig_14_score_dow_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_14_score_dow_trend.png)


![03_eda_fig_15_daily_score_trend.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_15_daily_score_trend.png)


![03_eda_fig_16_corr_heatmap.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_16_corr_heatmap.png)


![03_eda_fig_18_hour_day_heatmap.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/03_eda_fig_18_hour_day_heatmap.png)


![04_feature_fig_02_topic_entropy_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_02_topic_entropy_distribution.png)


![04_feature_fig_03_dominant_topic_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_03_dominant_topic_distribution.png)


![04_feature_fig_07_pred_prob_distribution.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_07_pred_prob_distribution.png)


![04_feature_fig_08_top_coefficients.png](../../Project%20Workspace/Supporting%20Materials/Report%20Sources/artifacts/figures/04_feature_fig_08_top_coefficients.png)
